<a href="https://colab.research.google.com/github/heoconngoc/Ruled-Based-A.I.-and-Deep-Learning/blob/main/Rule_Based_Tic_Tac_Toe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# =========================================================
# TITLE: Rule-Based Tic Tac Toe (1-step, 2-step, 3-step)
# =========================================================

"""
This notebook implements rule-based Tic Tac Toe agents:
- 1-step ahead
- 2-step ahead
- 3-step ahead

It includes:
- Console visualization
- Benchmark evaluation
"""

import random
import time

EMPTY = " "
X = "X"
O = "O"

In [3]:
# ---------------------------
# GAME LOGIC
# ---------------------------
def create_board():
    return [EMPTY]*9

def print_board(b):
    for i in range(0,9,3):
        print(b[i], "|", b[i+1], "|", b[i+2])
        if i < 6:
            print("---------")
    print()

def check_winner(b):
    wins = [(0,1,2),(3,4,5),(6,7,8),
            (0,3,6),(1,4,7),(2,5,8),
            (0,4,8),(2,4,6)]
    for i,j,k in wins:
        if b[i]!=EMPTY and b[i]==b[j]==b[k]:
            return b[i]
    return None

def get_moves(b):
    return [i for i in range(9) if b[i]==EMPTY]

def switch(p):
    return O if p==X else X

In [4]:
# ---------------------------
# RULE-BASED AGENTS
# ---------------------------

# 1-step: win or block
def rule_1_step(board, player):
    opponent = switch(player)

    # win
    for m in get_moves(board):
        b = board.copy()
        b[m] = player
        if check_winner(b) == player:
            return m

    # block
    for m in get_moves(board):
        b = board.copy()
        b[m] = opponent
        if check_winner(b) == opponent:
            return m

    return random.choice(get_moves(board))


# 2-step: simulate opponent response
def rule_2_step(board, player):
    opponent = switch(player)
    best_move = None

    for m in get_moves(board):
        b = board.copy()
        b[m] = player

        # if win immediately
        if check_winner(b) == player:
            return m

        safe = True
        for opp in get_moves(b):
            bb = b.copy()
            bb[opp] = opponent
            if check_winner(bb) == opponent:
                safe = False
        if safe:
            return m

    return random.choice(get_moves(board))

# 3-step: deeper reasoning
def rule_3_step(board, player):
    opponent = switch(player)

    best_move = None
    best_score = float('-inf')

    for m in get_moves(board):
        b = board.copy()
        b[m] = player

        if check_winner(b) == player:
            return m

        score = 0

        for opp in get_moves(b):
            bb = b.copy()
            bb[opp] = opponent

            if check_winner(bb) == opponent:
                score -= 10

            else:
                for m2 in get_moves(bb):
                    bbb = bb.copy()
                    bbb[m2] = player

                    if check_winner(bbb) == player:
                        score += 5
        if score > best_score:
            best_score = score
            best_move = m

    if best_move is not None:
        return best_move

    return random.choice(get_moves(board))

In [5]:
# ---------------------------
# GAME SIMULATION
# ---------------------------

def play_game(agentX, agentO, verbose=False):
    board = create_board()
    player = X

    while True:
        if player == X:
            move = agentX(board, X)
        else:
            move = agentO(board, O)

        board[move] = player

        if verbose:
            print_board(board)

        winner = check_winner(board)
        if winner:
            return 1 if winner==X else -1

        if not get_moves(board):
            return 0

        player = switch(player)

In [11]:
# ---------------------------
# BENCHMARK
# ---------------------------
results = []
for i in range(100):
    if i % 2 == 0:
        res = play_game(rule_2_step, rule_1_step)
    else:
        res = play_game(rule_1_step, rule_2_step)

    results.append(res)

print("Win:", results.count(1))
print("Loss:", results.count(-1))
print("Tie:", results.count(0))

Win: 16
Loss: 25
Tie: 59


In [13]:
# ---------------------------
# COMMON EVALUATION FRAMEWORK
# ---------------------------
def evaluate(agentA, agentB, games=100):
    results = []
    start = time.time()

    for i in range(games):
        if i % 2 == 0:
            res = play_game(agentA, agentB)
        else:
            res = play_game(agentB, agentA)
            res = -res

        results.append(res)

    end = time.time()

    print("Evaluation result:")
    print("Wins:", results.count(1))
    print("Losses:", results.count(-1))
    print("Ties:", results.count(0))
    print("Time:", round(end-start,4), "seconds")

# compare
evaluate(rule_3_step, rule_1_step)

Evaluation result:
Wins: 36
Losses: 44
Ties: 20
Time: 0.0741 seconds


In [12]:
evaluate(rule_3_step, rule_2_step)

Evaluation result:
Wins: 50
Losses: 0
Ties: 50
Time: 0.0844 seconds


Rule-based agents are extremely fast because they do not explore the full game tree. However, their decisions are limited and often suboptimal compared to search-based methods.